# Parkscraper
Scrape state parks data

In [ ]:
import json
import re
import requests
import time

from bs4 import BeautifulSoup
from datetime import datetime

In [32]:
# Create urls
STATE_BASE_URL = "https://www.parks.ca.gov"
STATE_PARKS_URL = BASE_URL + "/Find-a-Park"
BASE_URL, PARKS_URL

('https://www.parks.ca.gov', 'https://www.parks.ca.gov/Find-a-Park')

In [33]:
# Get all park links
def get_state_park_links():
    """ Fetch all California State Parks URLs. """
    try:
        response = requests.get(STATE_PARKS_URL)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        links = [
            BASE_URL + a["href"]
            for a in soup.select("div.park-display a")
            if a.get("href").startswith("/?page_id=")
        ]
        
        print(f"Found {len(links)} state parks!")
        return links
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error occurred: {e}")
    except requests.exception.RequestException as e:
        print(f"An error occurred: {e}")

In [36]:
# Get key-value pairs of information from state parks
def extract_key_value_pairs(soup_section):
    """ Extract key-value pairs from paragraphs and lists. """
    info = {}
    for p in soup_section.find_all("p"):
        text = p.get_text(" ", strip=True)
        if not text:
            continue
        strong = p.find("strong")
        if strong:
            label = strong.get_text(strip=True).rstrip(":")
            value = text.replace(label, "").strip(" :\n\t")
            if value:
                info[label] = value
        else:
            match = re.match(r"^([A-Z][A-Za-z\s]+):\s*(.+)$", text)
            if match:
                info[(match.group(1).strip())] = match.group(2).strip()
    return info

In [37]:
# Scrape one individual park
def scrape_state_park(url):
    """ Scrape a single CA State Park given a URL. """
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error occurred: {e}")
        return None
    except requests.exception.RequestException as e:
        print(f"An error occurred: {e}")
        return None
    
    soup = BeautifulSoup(response.text, "html.parser")

    # Create initial data dict
    data = {
        "URL": url,
        "Jurisdiction": "State"        
    }

    # Get park name
    name = soup.find("h1")
    data["Park Name"] = name.get_text(strip=True) if name else "Unknown"

    # Get page body
    main = soup.find("div", id="mainContent") or soup.find("div", id="PageContent") or soup

    paragraphs = main.find_all("p")

    # Get park descriptions
    desc = " ".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
        if not p.find("strong") and len(p.get_text(strip=True)) > 40
    )

    # Add description to data
    data["Description"] = desc.strip()

    # Get contacts/lists
    side = soup.find("div", id="sideContent")
    if side:
        for li in side.find_all("li"):
            text = li.get_text(" ", strip=True)
            if ":" in text:
                k, v = text.split(":", 1)
                data[k.strip()] = v

    return data

In [46]:
generate_file_name(scrape_state_park(BASE_URL + "/?page_id=531")["Park Name"])

'half_moon_bay_state_beach.txt'

In [45]:
# Helper function
def generate_file_name(title):
    """ Returns a file name with .txt extension given a title. """
    return "_".join(title.lower().split()) + ".txt"

In [63]:
def scrape_state_parks(output_file='../data/ca_state_parks.json',
                       limit_state = None,
                       limit_national = None,
                       delay = 1.0):
    """ Scrapes all CA State Parks and saves data into the output file. """
    
    state_park_links = get_state_park_links()

    results = []

    for i, link in enumerate(state_park_links):
        # If only scraping a subset (limit_state) of the park
        if limit_state and i >= limit_state:
            break
        park = scrape_state_park(link)
        results.append(park)
        time.sleep(delay)
    
    # Add scraping timestamp to each park data
    for prk in results:
        prk["Last Scraped"] = datetime.utcnow().isoformat()
    
    print(f"\n Total scraped: {len(results)} state parks!")

    # Save to output file
    if limit_state:
        output_file, ext = output_file.rsplit(".", maxsplit=1)
        output_file = output_file + f"_limit_{limit_state}" + "." + ext
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    print(f"Saved state parks dataset to {output_file}")

In [65]:
scrape_state_parks(limit_state=5)

Found 283 state parks!

 Total scraped: 5 state parks!
Saved state parks dataset to ../data/ca_state_parks_limit_5.json


In [7]:
# Get response
response = requests.get(PARKS_URL)
soup = BeautifulSoup(response.text, "html.parser")

In [29]:
sl = park_links[0]
presponse = requests.get(sl)
psoup = BeautifulSoup(presponse.text, "html.parser")
title = psoup.find("h1").text.strip() if psoup.find("h1") else "Unknown Park"
text_blocks = psoup.get_text(separator=" ", strip=True)

print(sl)
print(title)
print(text_blocks)

https://www.parks.ca.gov/?page_id=424
Admiral William Standley State Recreation Area
Admiral William Standley State Recreation Area Skip to Main Content About Us Contact Us FAQs Home About Us Contact Us Publications PRA Request Parks Director Public Notices Accessibility and Parks Organizational Structure Opportunities. Off-Highway Motor Vehicle Recreation Division Office of Historic Preservation Division of Boating and Waterways Commissions Find a Park Park Events Safety Tips Rules and Regulations Historic Sites Dogs in Parks Maps Travel Tips Live Video Gallery Weddings at State Parks Filming & Photography Reservations Reservations and Fees Sales Dates Availability Search Camping Information Park Passes Adventure Pass Manage Reservations Guided Tours Hearst Castle Tours What's New Parks Newsroom Parks Blog Significant Incidents Updates Frequently Asked Questions Jobs Job Postings Exam Announcements Careers Now Recruiting Seasonal and Part Time Law Enforcement Careers Volunteers Camp/P